In [2]:
import math
from collections import Counter

In [6]:
class SimpleBM25:
    def __init__(self, documents, k1 = 1.5, b= 0.75):
        self.documents = documents
        self.k1 = k1
        self.b = b

        self.tokenized_docs = [doc.lower().split() for doc in documents]

        self.doc_lengths = [len(doc) for doc in self.tokenized_docs]
        self.avg_doc_lengths = sum(self.doc_lengths)/len(self.doc_lengths)

        self.N = len(documents)

        self.df = {}

        for doc in self.tokenized_docs:
            unique_words = set(doc)

            for word in unique_words:
                self.df[word] = (self.df.get(word, 0) + 1)

    def idf(self, term):
        df = self.df.get(term, 0)

        return math.log((self.N - df + 0.5)/(df + 0.5) + 1)

    def score(self, query, doc_index):
        query_terms = query.lower().split()
        doc = self.tokenized_docs[doc_index]
        doc_counter = Counter(doc)

        score = 0

        for term in query_terms:
            tf = doc_counter[term]
            if tf == 0:
                continue
            idf = self.idf(term)

            num = tf * (self.k1  +1)

            deno = ( tf + self.k1*(1-self.b + self.b * len(doc)/self.avg_doc_lengths))

            score += idf * (num/deno)
        return score
    
    def search(self, query, top_k = 3):
        scores = []

        for i, doc in enumerate(self.documents):
            score = self.score(query,i)
            scores.append((score, doc))
        scores.sort(reverse=True)
        return scores[:top_k]

In [7]:
docs = [
    "Transformers use attention mechanism",
    "Football world cup analysis",
    "Attention mechanism improves language models",
    "Deep learning with transformers"
]

bm25 = SimpleBM25(docs)

results = bm25.search(
    "attention mechanism",
    top_k=3
)

for score, doc in results:
    print(round(score, 3), doc)

1.424 Transformers use attention mechanism
1.284 Attention mechanism improves language models
0 Football world cup analysis


In [1]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np

/Users/ratulsur/Desktop/all_data/GenAIand RAG/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# =====================================================
# SAMPLE USAGE
# =====================================================

documents = [

    "Transformers use attention mechanism",

    "Self attention is a key idea in transformers",

    "Attention improves language models",

    "Deep learning with transformers",

    "Football world cup analysis",

    "Neural network architectures",

    "Retrieval augmented generation systems",

    "Vector databases store embeddings",

    "Cross encoders improve reranking",

    "Cricket world cup predictions"
]

retriever = (
    HybridRetriever(
        documents,
        alpha=0.5
    )
)

query = (
    "attention mechanism in transformers"
)

results = retriever.retrieve(
    query
)

print("\nFINAL RESULTS\n")

for i, result in enumerate(
    results,
    start=1
):

    print(
        f"{i}. "
        f"CrossEncoder={result['cross_encoder_score']:.4f} | "
        f"Hybrid={result['hybrid_score']:.4f}"
    )

    print(
        result["document"]
    )

    print()

AttributeError: 'HybridRetriever' object has no attribute 'retrieve'

In [6]:
from rank_bm25 import BM25Okapi
from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)
import numpy as np


class HybridRetriever:

    def __init__(
        self,
        documents,
        alpha=0.5
    ):
        """
        alpha: BM25 weight
        (1-alpha): Dense retrieval weight
        """

        self.documents = documents
        self.alpha = alpha

        # BM25
        self.tokenized_docs = [
            doc.lower().split()
            for doc in documents
        ]

        self.bm25 = BM25Okapi(
            self.tokenized_docs
        )

        # Embedding Model
        self.embedding_model = (
            SentenceTransformer(
                "all-MiniLM-L6-v2"
            )
        )

        self.doc_embeddings = (
            self.embedding_model.encode(
                documents,
                normalize_embeddings=True,
                convert_to_numpy=True
            )
        )

        # Cross Encoder
        self.cross_encoder = (
            CrossEncoder(
                "cross-encoder/ms-marco-MiniLM-L-6-v2"
            )
        )

    # ======================================
    # NORMALIZATION
    # ======================================

    def _normalize(
        self,
        scores
    ):

        scores = np.array(scores)

        if scores.max() == scores.min():
            return np.ones_like(scores)

        return (
            scores - scores.min()
        ) / (
            scores.max() - scores.min()
        )

    # ======================================
    # HYBRID SEARCH
    # ======================================

    def hybrid_search(
        self,
        query,
        top_k=50
    ):

        query_tokens = (
            query.lower().split()
        )

        bm25_scores = (
            self.bm25.get_scores(
                query_tokens
            )
        )

        query_embedding = (
            self.embedding_model.encode(
                query,
                normalize_embeddings=True,
                convert_to_numpy=True
            )
        )

        dense_scores = np.dot(
            self.doc_embeddings,
            query_embedding
        )

        bm25_scores = (
            self._normalize(
                bm25_scores
            )
        )

        dense_scores = (
            self._normalize(
                dense_scores
            )
        )

        hybrid_scores = (
            self.alpha * bm25_scores
            +
            (1 - self.alpha)
            * dense_scores
        )

        ranked_indices = (
            np.argsort(
                hybrid_scores
            )[::-1]
        )

        results = []

        for idx in ranked_indices[:top_k]:

            results.append({

                "doc_id": idx,

                "document":
                    self.documents[idx],

                "embedding":
                    self.doc_embeddings[idx],

                "hybrid_score":
                    float(
                        hybrid_scores[idx]
                    ),

                "bm25_score":
                    float(
                        bm25_scores[idx]
                    ),

                "dense_score":
                    float(
                        dense_scores[idx]
                    )
            })

        return results

    # ======================================
    # MMR
    # ======================================

    def mmr(
        self,
        query,
        retrieved_docs,
        top_k=15,
        lambda_parameter=0.7
    ):

        query_embedding = (
            self.embedding_model.encode(
                query,
                normalize_embeddings=True,
                convert_to_numpy=True
            )
        )

        doc_embeddings = np.array([

            d["embedding"]

            for d in retrieved_docs

        ])

        query_similarities = np.dot(
            doc_embeddings,
            query_embedding
        )

        selected = []

        remaining = list(
            range(
                len(retrieved_docs)
            )
        )

        # First document

        first_idx = np.argmax(
            query_similarities
        )

        selected.append(
            first_idx
        )

        remaining.remove(
            first_idx
        )

        # Greedy MMR

        while (
            len(selected) < top_k
            and remaining
        ):

            best_score = -np.inf
            best_doc = None

            for idx in remaining:

                relevance = (
                    query_similarities[idx]
                )

                redundancy = max(

                    np.dot(
                        doc_embeddings[idx],
                        doc_embeddings[s]
                    )

                    for s in selected
                )

                mmr_score = (

                    lambda_parameter
                    * relevance

                    -

                    (
                        1 -
                        lambda_parameter
                    )
                    * redundancy
                )

                if (
                    mmr_score
                    >
                    best_score
                ):

                    best_score = mmr_score
                    best_doc = idx

            selected.append(
                best_doc
            )

            remaining.remove(
                best_doc
            )

        return [

            retrieved_docs[i]

            for i in selected

        ]

    # ======================================
    # CROSS ENCODER
    # ======================================

    def rerank(
        self,
        query,
        docs,
        top_k=5
    ):

        pairs = [

            (
                query,
                d["document"]
            )

            for d in docs

        ]

        scores = (
            self.cross_encoder.predict(
                pairs
            )
        )

        for doc, score in zip(
            docs,
            scores
        ):

            doc[
                "cross_encoder_score"
            ] = float(score)

        docs.sort(

            key=lambda x:
            x[
                "cross_encoder_score"
            ],

            reverse=True
        )

        return docs[:top_k]

In [8]:
documents = [

    "Transformers use attention mechanism",

    "Self attention is a key idea in transformers",

    "Attention improves language models",

    "Deep learning with transformers",

    "Football world cup analysis",

    "Neural network architectures",

    "Retrieval augmented generation systems",

    "Vector databases store embeddings",

    "Cross encoders improve reranking",

    "Cricket world cup predictions"
]

In [9]:
retriever = HybridRetriever(
    documents,
    alpha=0.5
)

In [10]:
query = "attention mechanism in transformers"

In [11]:
hybrid_results = retriever.hybrid_search(
    query,
    top_k=5
)

print("\nHYBRID RESULTS")
print("-" * 50)

for result in hybrid_results:

    print(
        f"Hybrid={result['hybrid_score']:.4f}"
    )

    print(
        result["document"]
    )

    print()


HYBRID RESULTS
--------------------------------------------------
Hybrid=1.0000
Transformers use attention mechanism

Hybrid=0.7718
Self attention is a key idea in transformers

Hybrid=0.3744
Deep learning with transformers

Hybrid=0.3513
Attention improves language models

Hybrid=0.1400
Neural network architectures



In [12]:
mmr_results = retriever.mmr(
    query,
    hybrid_results,
    top_k=3,
    lambda_parameter=0.7
)

print("\nMMR RESULTS")
print("-" * 50)

for result in mmr_results:

    print(
        result["document"]
    )


MMR RESULTS
--------------------------------------------------
Transformers use attention mechanism
Self attention is a key idea in transformers
Deep learning with transformers


In [13]:
reranked_results = retriever.rerank(
    query,
    mmr_results,
    top_k=3
)

print("\nRERANKED RESULTS")
print("-" * 50)

for result in reranked_results:

    print(
        f"CrossEncoder={result['cross_encoder_score']:.4f}"
    )

    print(
        result["document"]
    )

    print()


RERANKED RESULTS
--------------------------------------------------
CrossEncoder=8.3577
Transformers use attention mechanism

CrossEncoder=4.6747
Self attention is a key idea in transformers

CrossEncoder=-5.8352
Deep learning with transformers

